# Anthony Qwen V1 Routing Benchmark
Run this notebook on a Colab T4 GPU after the benchmark is frozen. It evaluates the existing persona/intent Qwen model on the exact V1 benchmark and writes a prediction export that the repository adapter can score as routing complexity.


In [ ]:
!pip -q install unsloth pandas
!git clone -q https://github.com/Shivansh-Sahni/QueryIntentResolver.git
%cd QueryIntentResolver


In [ ]:
from unsloth import FastLanguageModel
import pandas as pd
import re, time, torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='TheCupNoodle/query-intent-classifier',
    max_seq_length=512,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)


In [ ]:
benchmark = pd.read_csv('v1/generated/benchmark/benchmark_queries.csv')
len(benchmark), benchmark.head()


In [ ]:
def parse_field(text, field):
    match = re.search(rf'{field}\s*:\s*([a-zA-Z0-9 _-]+)', text, flags=re.I)
    if not match:
        return ''
    return re.sub(r'[^a-z0-9]+', '_', match.group(1).splitlines()[0].strip().lower()).strip('_')

def classify(query):
    prompt = f'''### Instruction:
Classify this college-related query.
### Input:
{query}
### Response:'''
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.no_grad():
        output = model.generate(
            **inputs, max_new_tokens=50, do_sample=False,
            return_dict_in_generate=True, output_scores=True,
        )
    torch.cuda.synchronize()
    latency_ms = (time.perf_counter() - started) * 1000
    generated = output.sequences[0, inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip()
    confidence = 0.5
    if output.scores:
        transition = model.compute_transition_scores(output.sequences, output.scores, normalize_logits=True)
        confidence = float(torch.exp(transition[0].mean()).clamp(0, 1).cpu())
    return response, parse_field(response, 'persona'), parse_field(response, 'intent'), confidence, latency_ms


In [ ]:
records = []
for i, row in benchmark.iterrows():
    response, persona, intent, confidence, latency_ms = classify(row['query_text'])
    records.append({
        'benchmark_row_id': row['benchmark_row_id'],
        'query_id': row['query_id'],
        'query_text': row['query_text'],
        'response': response,
        'predicted_persona': persona,
        'predicted_intent': intent,
        'confidence': confidence,
        'latency_ms': latency_ms,
        'estimated_cost_usd': 0.0,
    })
    if (i + 1) % 25 == 0:
        print(f'{i + 1}/{len(benchmark)}')
predictions = pd.DataFrame(records)
predictions.to_csv('qwen_v1_benchmark_predictions.csv', index=False)
predictions.head()


Download `qwen_v1_benchmark_predictions.csv`, place it in the repository, then run:

```bash
python v1/scripts/adapt_qwen_predictions.py \
  --benchmark v1/generated/benchmark/benchmark_gold.csv \
  --qwen-predictions qwen_v1_benchmark_predictions.csv \
  --output-dir v1/generated/models/qwen
```
